# Phase 1 : OFFLINE & CALIBRATION

In [1]:
import os
from datasets import load_dataset, load_from_disk
import pandas as pd
from IPython.display import display, Markdown

from encoders import build_encoder
from utils_indexation import build_indices
import config

c:\Users\mario\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\mario\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\mario\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framewo

ModuleNotFoundError: No module named 'open_clip'

## Configuration

In [ ]:
ENCODER_REGISTRY = {
    #"BLIP":      (build_encoder("blip"),      256),
    #"SigLIP":    (build_encoder("siglip"),    1152),
    #"DINO_BERT": (build_encoder("dino_bert"), 512),
    #"CONVNEXT_CLIP":(build_encoder("convnext_clip"), 1024),
    #"EVA_CLIP":   (build_encoder("eva_clip"),  768),
}

In [ ]:
DATASETS_CONFIG = {
    "flickr30k": {
        "hf_path": "nlphuji/flickr30k",
        "custom_split": True,
        "image_col": "image",
        "caption_col": "caption"
    },
    "flickr8k": {
        "hf_path": "dandelin/flickr8k",
        "custom_split": False,
        "image_col": "image",
        "caption_col": "captions"
    },
}

config_ds = DATASETS_CONFIG[config.DATASET_NAME]

## Chargement des Données

In [ ]:
print(f"Chargement du dataset : {config.DATASET_NAME}...")

if os.path.exists(f"{config.RAW_DATA_DIR}/train"):
    print("Chargement depuis le disque local...")
    train_dataset = load_from_disk(f"{config.RAW_DATA_DIR}/train")
    test_dataset  = load_from_disk(f"{config.RAW_DATA_DIR}/test")
    val_dataset   = load_from_disk(f"{config.RAW_DATA_DIR}/val")
else:
    print("Téléchargement et formatage depuis HuggingFace...")
    if config_ds["custom_split"]:
        # Logique spécifique (ex: Flickr30k de nlphuji)
        raw_dataset = load_dataset(config_ds["hf_path"], trust_remote_code=True)['test']
        train_dataset = raw_dataset.filter(lambda x: x['split'] == 'train')
        test_dataset  = raw_dataset.filter(lambda x: x['split'] == 'test')
        val_dataset   = raw_dataset.filter(lambda x: x['split'] == 'val')
    else:
        # Logique standard HuggingFace (ex: Flickr8k)
        raw_dataset = load_dataset(config_ds["hf_path"], trust_remote_code=True)
        train_dataset = raw_dataset['train']
        test_dataset  = raw_dataset['test']
        # Gérer le nom du split de validation (parfois 'validation', parfois 'val')
        val_split_name = 'validation' if 'validation' in raw_dataset else 'val'
        val_dataset   = raw_dataset[val_split_name]
    
    # Sauvegarde pour la prochaine fois
    train_dataset.save_to_disk(f"{config.RAW_DATA_DIR}/train")
    test_dataset.save_to_disk(f"{config.RAW_DATA_DIR}/test")
    val_dataset.save_to_disk(f"{config.RAW_DATA_DIR}/val")

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

## Calibration

In [ ]:
needs_calibration = any(encoder.needs_calibration for encoder, _ in ENCODER_REGISTRY.values())

if needs_calibration:
    cal_images, cal_texts = [], []
    
    for i, item in enumerate(train_dataset):
        if len(cal_texts) >= 10000: break
        captions = item[config_ds['caption_col']] if isinstance(item[config_ds['caption_col']], list) else [item[config_ds['caption_col']]]
        for cap in captions:
            cal_images.append(item[config_ds['image_col']])
            cal_texts.append(cap)

    print(f"Paires prêtes : {len(cal_texts)}")
    
    for name, (encoder, _) in ENCODER_REGISTRY.items():
        if encoder.needs_calibration:
            encoder.calibrate(cal_images, cal_texts, batch_size=32, epochs=20)
            
    print("\nCalibration terminée avec succès !")

## Indexation et Sauvegarde

In [ ]:
print("\n--- Indexation Validation ---")
val_corpus = build_indices(
    val_dataset,
    ENCODER_REGISTRY,
    image_field   = config_ds['image_col'],
    caption_field = config_ds['caption_col'],
    batch_size    = 32,
    save_dir      = config.INDEX_DIR,
    prefix        = "val_",
    store_vectors = True,
)

print("\n--- Indexation Test ---")
test_corpus = build_indices(
    test_dataset,
    ENCODER_REGISTRY,
    image_field   = config_ds['image_col'],
    caption_field = config_ds['caption_col'],
    batch_size    = 64,
    save_dir      = config.INDEX_DIR,
    prefix        = "test_",
    store_vectors = True,
)

## Résultats des Temps d'Indexation (Val + Test)

In [ ]:
# 1. Agrégation des temps (Validation + Test)
total_timing = {}
for name in ENCODER_REGISTRY:
    total_timing[name] = {
        "Images (s)": val_corpus.timing_stats[name]["Images (s)"] + test_corpus.timing_stats[name]["Images (s)"],
        "Textes (s)": val_corpus.timing_stats[name]["Textes (s)"] + test_corpus.timing_stats[name]["Textes (s)"],
        "Total (s)": val_corpus.timing_stats[name]["Total (s)"] + test_corpus.timing_stats[name]["Total (s)"]
    }

# 2. Création du DataFrame
df_times = pd.DataFrame.from_dict(total_timing, orient='index')

# 3. Formatage pour le Markdown (On met en gras la valeur la plus BASSE)
df_formatted_times = df_times.copy()
for col in df_formatted_times.columns:
    min_val = df_times[col].min() # <-- On cherche le minimum !
    df_formatted_times[col] = df_times[col].apply(
        lambda x: f"**{x:.2f}**" if x == min_val else f"{x:.2f}"
    )

# 4. Affichage
print("\n" + "="*60)
print("RÉSULTATS - TEMPS D'INDEXATION EN SECONDES (Val + Test)")
print("="*60)
display(Markdown(df_formatted_times.to_markdown()))

# 5. Sauvegarde
chemin_csv_temps = f"{config.INDEX_DIR}/temps_indexation.csv"
chemin_md_temps = f"{config.INDEX_DIR}/temps_indexation.md"

df_times.to_csv(chemin_csv_temps, float_format="%.2f")
df_formatted_times.to_markdown(chemin_md_temps)

print(f"Temps d'indexation sauvegardés avec succès dans :")
print(f"   - {chemin_csv_temps}")
print(f"   - {chemin_md_temps}")